# Create, evaluate, and deploy an AI agent

This notebook walks through building an AI agent that combines a retrieval tool and a general-purpose tool, testing it, and packaging it as an MLflow model. It uses a pre-chunked subset of Databricks documentation as the example dataset.

This version is adapted to run locally in VS Code or Jupyter with a normal Python kernel. Here is what changed from the original Databricks notebook:

- The LLM is called through an OpenAI-compatible API instead of a Databricks model serving endpoint. Swap in any provider you like (OpenAI, Azure OpenAI, a local Ollama server, etc).
- Unity Catalog is not available outside Databricks, so the `tfidf_keywords` tool is defined as a plain LangChain tool instead of being registered as a UC function.
- Databricks Agent Evaluation (`databricks.agents.evals`) and Unity Catalog model deployment (`databricks.agents.deploy`) are managed Databricks features with no local equivalent. Those two sections are kept as reference code in markdown cells near the end, clearly marked, so the notebook still runs cleanly top to bottom.

## Install dependencies

This installs everything needed to run the notebook locally: MLflow for tracing/model packaging, LangChain and LangGraph for the agent, LangChain's OpenAI integration for the LLM call, and scikit-learn for the TF-IDF tools.

If this is the first time you're installing these packages in this environment, restart the kernel after this cell finishes before running the rest of the notebook.

In [ ]:
%pip install -U -qqqq mlflow langchain langgraph==0.3.4 langchain-openai pydantic scikit-learn pandas


## Set up the LLM connection

Set your API key below, or set the `OPENAI_API_KEY` environment variable before starting Jupyter and skip this step. This example uses OpenAI, but any OpenAI-compatible endpoint works if you also pass `base_url` to `ChatOpenAI`.

In [ ]:
import os

# TODO: set your API key here, or set the OPENAI_API_KEY environment variable before starting Jupyter
os.environ.setdefault("OPENAI_API_KEY", "<YOUR_OPENAI_API_KEY>")

from langchain_openai import ChatOpenAI

# TODO: replace with the model you want to use
MODEL_NAME = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.01)


## Load the dataset

This pulls a pre-chunked subset of Databricks documentation, used later by the retrieval tool.

In [ ]:
import pandas as pd

databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)
parsed_docs_df.head()


## Create a general-purpose tool

The original notebook registers this function as a Unity Catalog function so it can be shared across users and workspaces. Locally, there is no Unity Catalog, so it's just defined as a plain LangChain tool.

In [ ]:
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer


@tool
def tfidf_keywords(text: str) -> list[str]:
    """
    Extracts keywords from the provided text using TF-IDF.

    Args:
        text (string): Input text.
    Returns:
        list[str]: List of extracted keywords in ascending order of importance.
    """
    top_n = 5
    vectorizer = TfidfVectorizer(stop_words="english")  # new vectorizer for this call
    query_tfidf = vectorizer.fit_transform([text])
    scores = query_tfidf.toarray()[0]
    indices = scores.argsort()[-top_n:][::-1]  # top N keyword indices
    return [vectorizer.get_feature_names_out()[i] for i in indices if scores[i] > 0]


In [ ]:
print(tfidf_keywords)
tfidf_keywords.invoke({"text": "The quick brown fox jumped over the lazy brown dog."})


## Create the retrieval tool

This builds a TF-IDF index over the documentation and exposes it as a tool the agent can call. The `mlflow.trace` decorator records the call as a retriever span, so it shows up in the trace UI.

In [ ]:
from typing import Any

import mlflow
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer

documents = parsed_docs_df
doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """gets relevant documents for the query"""
    query_tfidf = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_tfidf.T).toarray().flatten()
    ranked_docs = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    result = []
    for idx, score in ranked_docs[:top_n]:
        row = documents.iloc[idx]
        content = row["content"]
        doc_entry = {
            "page_content": content,
            "metadata": {
                "doc_uri": row["doc_uri"],
                "score": score,
            },
        }
        result.append(doc_entry)
    return result


## Build the agent graph

This wires the LLM and tools together with LangGraph. The agent calls a tool when the model asks for one, then loops back until it has a final answer.

In [ ]:
from typing import Optional, Sequence, Union

from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def routing_logic(state: ChatAgentState):
        last_message = state["messages"][-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if agent_prompt:
        system_message = {"role": "system", "content": agent_prompt}
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        routing_logic,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


## Test the agent

`mlflow.langchain.autolog()` traces every LangChain and LangGraph call automatically, no extra code needed.

In [ ]:
import mlflow

mlflow.langchain.autolog()

agent = create_tool_calling_agent(llm, tools=[tfidf_keywords, find_relevant_documents])
agent.invoke({"messages": [{"role": "user", "content": "What are the keywords for the sentence: 'the quick brown fox jumped over the lazy brown dog'?"}]})


## Wrap the agent in MLflow's ChatAgent interface

Wrapping the agent this way gives it a standard `predict` signature, which makes it easier to log, evaluate, and serve later.

In [ ]:
from typing import Any, Optional

from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatContext


class DocsAgent(ChatAgent):
    def __init__(self, agent):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # ChatAgent has a built-in helper to convert framework-specific messages to a plain dict
        request = {"messages": self._convert_messages_to_dict(messages)}

        output = agent.invoke(request)
        # output is already a ChatAgentResponse, but we wrap it again to make the signature explicit
        return ChatAgentResponse(**output)


In [ ]:
AGENT = DocsAgent(agent=agent)
AGENT.predict({"messages": [{"role": "user", "content": "What is DLT in Databricks?"}]})


## Make the agent configurable

Moving settings into a config dict makes it easy to swap the model, prompt, or tools without touching code. This is the version we'll package up as a standalone script next.

In [ ]:
from mlflow.models import ModelConfig

baseline_config = {
    "model_name": MODEL_NAME,
    "temperature": 0.01,
    "max_tokens": 1000,
    "system_prompt": """You are a helpful assistant that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.

    You answer questions using a set of tools. If needed, you ask the user follow-up questions to clarify their request.
    """,
}


class DocsAgent(ChatAgent):
    def __init__(self):
        self.config = ModelConfig(development_config=baseline_config)
        self.agent = self._build_agent_from_config()

    def _build_agent_from_config(self):
        temperature = self.config.get("temperature")
        max_tokens = self.config.get("max_tokens")
        system_prompt = self.config.get("system_prompt")
        model_name = self.config.get("model_name")

        llm = ChatOpenAI(model=model_name, temperature=temperature, max_tokens=max_tokens)
        agent = create_tool_calling_agent(
            llm, tools=[tfidf_keywords, find_relevant_documents], agent_prompt=system_prompt
        )
        return agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # ChatAgent has a built-in helper to convert framework-specific messages to a plain dict
        request = {"messages": self._convert_messages_to_dict(messages)}

        output = self.agent.invoke(request)
        return ChatAgentResponse(**output)


agent = DocsAgent()
agent.predict({"messages": [{"role": "user", "content": "What is DLT"}]})


## Package the agent as a script

MLflow logs agents from a standalone Python file rather than notebook cells. Writing it out here keeps the file in sync with what was just tested above, and drops the notebook-only setup (API key placeholder aside).

In [ ]:
%%writefile getting_started_agent.py
import os
from typing import Any, Optional, Sequence, Union

import mlflow
import pandas as pd
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.models import ModelConfig
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatContext
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: set your API key here, or set the OPENAI_API_KEY environment variable before starting Jupyter
os.environ.setdefault("OPENAI_API_KEY", "<YOUR_OPENAI_API_KEY>")

databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)

documents = parsed_docs_df
doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """gets relevant documents for the query"""
    query_tfidf = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_tfidf.T).toarray().flatten()
    ranked_docs = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    result = []
    for idx, score in ranked_docs[:top_n]:
        row = documents.iloc[idx]
        content = row["content"]
        doc_entry = {
            "page_content": content,
            "metadata": {
                "doc_uri": row["doc_uri"],
                "score": score,
            },
        }
        result.append(doc_entry)
    return result


@tool
def tfidf_keywords(text: str) -> list[str]:
    """
    Extracts keywords from the provided text using TF-IDF.

    Args:
        text (string): Input text.
    Returns:
        list[str]: List of extracted keywords in ascending order of importance.
    """
    top_n = 5
    vectorizer = TfidfVectorizer(stop_words="english")  # new vectorizer for this call
    query_tfidf = vectorizer.fit_transform([text])
    scores = query_tfidf.toarray()[0]
    indices = scores.argsort()[-top_n:][::-1]  # top N keyword indices
    return [vectorizer.get_feature_names_out()[i] for i in indices if scores[i] > 0]


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def routing_logic(state: ChatAgentState):
        last_message = state["messages"][-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if agent_prompt:
        system_message = {"role": "system", "content": agent_prompt}
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        routing_logic,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class DocsAgent(ChatAgent):
    def __init__(self, config, tools):
        # Load config. When this agent is deployed to model serving, this config is
        # replaced with the one passed to mlflow.pyfunc.log_model(model_config=...)
        self.config = ModelConfig(development_config=config)
        self.tools = tools
        self.agent = self._build_agent_from_config()

    def _build_agent_from_config(self):
        llm = ChatOpenAI(
            model=self.config.get("model_name"),
            temperature=self.config.get("temperature"),
            max_tokens=self.config.get("max_tokens"),
        )
        agent = create_tool_calling_agent(
            llm,
            tools=self.tools,
            agent_prompt=self.config.get("system_prompt"),
        )
        return agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # ChatAgent has a built-in helper to convert framework-specific messages to a plain dict
        request = {"messages": self._convert_messages_to_dict(messages)}

        output = self.agent.invoke(request)
        return ChatAgentResponse(**output)


# TODO: replace with the model you want to use
MODEL_NAME = "gpt-4o-mini"

baseline_config = {
    "model_name": MODEL_NAME,
    "temperature": 0.01,
    "max_tokens": 1000,
    "system_prompt": """You are a helpful assistant that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.

    You answer questions using a set of tools. If needed, you ask the user follow-up questions to clarify their request.
    """,
}

tools = [find_relevant_documents, tfidf_keywords]

AGENT = DocsAgent(baseline_config, tools)
mlflow.models.set_model(AGENT)


If you already imported `getting_started_agent` earlier in this session, restart the kernel (or use `importlib.reload`) so the next cell picks up the file version instead of a cached module.

In [ ]:
from getting_started_agent import AGENT

AGENT.predict({"messages": [{"role": "user", "content": "What is DLT"}]})


## Log the agent with MLflow

This logs the agent as an MLflow model using your local MLflow tracking store. The original notebook also attaches Databricks resource references (serving endpoint, Unity Catalog function) so a deployed model can authenticate to those resources automatically; that part is skipped here since it only applies to Databricks model serving.

In [ ]:
import mlflow
from getting_started_agent import MODEL_NAME, baseline_config

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        python_model="getting_started_agent.py",
        name="agent",
        model_config=baseline_config,
        pip_requirements=[
            "mlflow",
            "langchain",
            "langgraph==0.3.4",
            "langchain-openai",
            "pydantic",
            "scikit-learn",
            "pandas",
        ],
        input_example={
            "messages": [{"role": "user", "content": "What is lakehouse monitoring?"}]
        },
    )

print(model_info.model_uri)


## Databricks-specific: Agent Evaluation

The rest of the original notebook uses Databricks Agent Evaluation (`databricks.agents.evals`) to generate a synthetic evaluation set and score the agent with Mosaic AI Agent Evaluation metrics. This is a managed Databricks feature with no local equivalent, so it will not run outside a Databricks workspace. The original code is kept below for reference, it is not an executable cell.

```python
import pandas as pd
from databricks.agents.evals import generate_evals_df

agent_description = """
The agent is a RAG chatbot that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.
"""
question_guidelines = """
# User personas
- A developer who is new to the Databricks platform
- An experienced, highly technical Data Scientist or Data Engineer

# Example questions
- what API lets me parallelize operations over rows of a delta table?
- Which cluster settings will give me the best performance when using Spark?

# Additional Guidelines
- Questions should be succinct, and human-like
"""

num_evals = 25
evals = generate_evals_df(
    docs=parsed_docs_df[:500],
    num_evals=num_evals,
    agent_description=agent_description,
    question_guidelines=question_guidelines,
)
display(evals)
```

```python
from databricks.agents.evals import metric

@metric
def uses_keywords_and_retriever(request, trace):
    retriever_spans = trace.search_spans(span_type="RETRIEVER")
    keyword_tool_spans = trace.search_spans(name="tfidf_keywords")
    return len(keyword_tool_spans) > 0 and len(retriever_spans) > 0
```

```python
# Note: this needs a provisioned throughput Foundation Model API endpoint for a higher rate limit on Databricks
with mlflow.start_run(run_name="my_agent"):
    eval_results = mlflow.evaluate(
        data=evals,
        model=model_info.model_uri,
        model_type="databricks-agent",
        extra_metrics=[uses_keywords_and_retriever],
    )
```

If you want to check the agent's quality locally instead, put together a small set of questions by hand and either call `AGENT.predict(...)` on each one and read the answers yourself, or score them with a plain `mlflow.evaluate(model_type="question-answering", ...)` run.

## Databricks-specific: register and deploy the agent

Registering to Unity Catalog and deploying with the Mosaic AI Agent Framework (`databricks.agents.deploy`) are also Databricks-only. They create a serving endpoint and review app that only exist inside a Databricks workspace. The original code is kept below for reference, it is not an executable cell.

```python
import mlflow
from databricks import agents

mlflow.set_registry_uri("databricks-uc")

# TODO: fill in your catalog and schema name
catalog = "your_catalog"
schema = "your_schema"
UC_MODEL_NAME = f"{catalog}.{schema}.getting_started_agent"

uc_registered_model_info = mlflow.register_model(
    model_uri=model_info.model_uri, name=UC_MODEL_NAME
)
deployment_info = agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, deploy_feedback_model=False)
```

To serve this agent outside Databricks, register it with a plain MLflow Model Registry instead (drop the `databricks-uc` registry URI) and serve it locally with:

```bash
mlflow models serve -m <model_uri> -p 5000
```